In [1]:
import random
import requests
import pandas as pd
import json

# CRAWL DATA

In [2]:
import requests
import json

url = "https://dummyjson.com/products"

reponse = requests.get(url)

if reponse.status_code == 200:
    data = reponse.json()

    with open("products.json","w", encoding="utf-8") as f:
        json.dump(data["products"], f, indent=4)

        print("Crawl products!")
else:
    print("Error:",reponse.status_code)

Crawl products!


SPARK chỉ đọc được file json dạng này: 
    
    {"id":1,"title":"abc"}
    {"id":2,"title":"xyz"}
nên phải đổi sang định dạng đó

In [3]:
import json

with open("products.json") as f:
    data = json.load(f)

with open("products_final.json", "w") as f:
    for item in data:
        f.write(json.dumps(item) + "\n")

# SPARK SETUP

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Ecommerce Pipeline") \
    .getOrCreate()

In [5]:
spark

# LOAD DATA

In [6]:
df = spark.read.json("products_final.json")
df.show(10)
df.printSchema()


+------------------+---------------+----------+--------------------+--------------------+------------------+---+--------------------+--------------------+--------------------+------+------+--------------------+--------------------+--------------------+---------------+-----+--------------------+--------------------+--------------------+-------------------+------+
|availabilityStatus|          brand|  category|         description|          dimensions|discountPercentage| id|              images|                meta|minimumOrderQuantity| price|rating|        returnPolicy|             reviews| shippingInformation|            sku|stock|                tags|           thumbnail|               title|warrantyInformation|weight|
+------------------+---------------+----------+--------------------+--------------------+------------------+---+--------------------+--------------------+--------------------+------+------+--------------------+--------------------+--------------------+---------------+--

# CONCAT DATA

In [16]:
df_dirty = spark.read.json('dirty_data.json')
df_products = spark.read.json('products_final.json')

from pyspark.sql.functions import lit

# Lấy danh sách cột của bảng to
all_columns = df_products.columns

# Thêm những cột thiếu vào bảng nhỏ, gán giá trị None (Null)
for col_name in all_columns:
    if col_name not in df_dirty.columns:
        df_dirty = df_dirty.withColumn(col_name, lit(None))

# Đảm bảo thứ tự cột khớp nhau trước khi union
df_final = df_dirty.select(all_columns).union(df_products.select(all_columns))

# coalesce(1) ép Spark gộp tất cả dữ liệu về 1 phân vùng duy nhất
df_final.coalesce(1).write.mode('overwrite').json('products_dirty_final.json')

In [ ]:
from pyspark.sql.functions import col, lit

# --- BƯỚC 1: ĐỒNG BỘ SCHEMA ---
target_schema = product_data.schema

for field in target_schema:
    col_name = field.name
    col_type = field.dataType
    
    if col_name not in dirty_data.columns:
        # Ép kiểu từ void sang kiểu dữ liệu thực tế của product_data
        dirty_data = dirty_data.withColumn(col_name, lit(None).cast(col_type))
    else:
        dirty_data = dirty_data.withColumn(col_name, col(col_name).cast(col_type))

# --- BƯỚC 2: NỐI BẢNG (UNION) ---
df_final = dirty_data.select(product_data.columns).union(product_data)

# --- BƯỚC 3: LƯU VÀO FILE MỚI ---
# .coalesce(1) giúp gom dữ liệu về 1 file duy nhất thay vì chia nhỏ thành nhiều file
# .mode('overwrite') sẽ ghi đè nếu file/thư mục đã tồn tại
df_final.coalesce(1).write.mode('overwrite').json('merged_products_full.json')

print("Đã lưu file thành công tại thư mục: merged_products_full.json")

In [17]:
df_dirty.printSchema()

root
 |-- category: string (nullable = true)
 |-- id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- stock: long (nullable = true)
 |-- title: string (nullable = true)
 |-- availabilityStatus: void (nullable = true)
 |-- brand: void (nullable = true)
 |-- description: void (nullable = true)
 |-- dimensions: void (nullable = true)
 |-- discountPercentage: void (nullable = true)
 |-- images: void (nullable = true)
 |-- meta: void (nullable = true)
 |-- minimumOrderQuantity: void (nullable = true)
 |-- rating: void (nullable = true)
 |-- returnPolicy: void (nullable = true)
 |-- reviews: void (nullable = true)
 |-- shippingInformation: void (nullable = true)
 |-- sku: void (nullable = true)
 |-- tags: void (nullable = true)
 |-- thumbnail: void (nullable = true)
 |-- warrantyInformation: void (nullable = true)
 |-- weight: void (nullable = true)

